# TP — Comprendre et dépasser Mem0

## Règles du jeu

Ce notebook est un **squelette**, pas une solution. Chaque cellule de code marquée `# TODO` est à écrire soi-même. Pour chaque item, trois choses sont attendues :

1. **Du code commenté** — expliquer en commentaire *pourquoi* chaque étape est faite, pas seulement *ce qui* est fait.
2. Une cellule **🔍 Constat** — ce qui est observé factuellement en exécutant le code (sorties, comportements, différences).
3. Une cellule **💬 Interprétation** — pourquoi ce comportement se produit, en le reliant au fonctionnement de Mem0 (extraction, ADD/UPDATE/DELETE/NOOP, recherche vectorielle, etc.).

Un seul exemple complet est donné dans l'item 1, pour montrer le pattern attendu — le reste est à construire.

## Plan

| Item | Contenu |
|---|---|
| 0 | Installation et configuration |
| 1 | Mem0 : premier contact (exemple guidé) |
| 2 | ADD / UPDATE / DELETE / NOOP |
| 3 | Persistance single-hop |
| 4 | Mémoire multi-hop et temporelle |
| 5 | Mini LLM-as-a-Judge |
| 6 | Mémoire en graphe (Mem0ᵍ) (bonus) |
| 7 | Intégration LangChain |
| 8 | Agent LangGraph avec mémoire |
| 9 | Connexion d'un outil (agent SAV Beqo) |
| 10 | Mesure latence / tokens, comparaison à un baseline |
| 11 | Bilan et pistes pour aller plus loin |


## Item 0 : Installation et configuration


In [ ]:
!pip install -q mem0ai openai langchain langchain-openai langgraph requests tiktoken matplotlib


In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Ta clé OpenAI API : ")

print("Clé chargée :", bool(os.environ.get("OPENAI_API_KEY")))


## Item 1 : Mem0 : premier contact (exemple guidé)

Voici le **seul** exemple entièrement écrit du notebook. À étudier attentivement : c'est le pattern à réutiliser (et adapter) pour tous les items suivants.


In [ ]:
from mem0 import Memory

# On configure Mem0 : quel LLM extrait/gère les souvenirs, quel modèle transforme le texte en vecteurs.
config = {
    "llm": {"provider": "openai", "config": {"model": "gpt-4o-mini", "temperature": 0.0}},
    "embedder": {"provider": "openai", "config": {"model": "text-embedding-3-small"}},
}
memory = Memory.from_config(config)

# On ajoute un premier souvenir pour un utilisateur donné.
resultat = memory.add("J'habite à Cotonou et je travaille dans la logistique.", user_id="demo_user")

# On inspecte ce que Mem0 a réellement fait avec ce message.
print(resultat)


**🔍 Constat (exemple résolu)** — La sortie contient une liste d'événements avec un champ `event`. Dans mon cas, j'observe un événement de type `ADD`, avec le texte extrait ("habite à Cotonou", "travaille dans la logistique" — éventuellement scindé en deux faits distincts).

**💬 Interprétation (exemple résolu)** — Mem0 n'a pas stocké ma phrase telle quelle : il l'a fait passer par sa phase d'extraction (un LLM qui isole les faits saillants), puis par sa phase de mise à jour, qui a comparé ces faits candidats à une mémoire vide — donc rien à contredire ni enrichir, d'où `ADD`.

## Item 2 : Observer ADD / UPDATE / DELETE / NOOP

C'est maintenant à toi de jouer, sans exemple pré-rempli.

**Consignes :**
1. Choisis un `user_id` de ton choix.
2. Ajoute un premier fait sur cet utilisateur (n'importe quel sujet : une préférence, une info personnelle...).
3. Ajoute un deuxième message qui **enrichit** ce fait (sans le contredire) — observe si Mem0 fait un `UPDATE`.
4. Ajoute un troisième message qui **contredit clairement** le premier fait — observe le `DELETE`/`UPDATE`.
5. Ajoute un quatrième message qui ne dit **rien de nouveau** (répète une info déjà connue, autrement formulée) — observe le `NOOP`.

Commente chaque appel : pourquoi tu t'attends à tel comportement avant même d'exécuter.


In [ ]:
# TODO — étape 1 et 2 : premier fait
# user_id = "..."
# resultat_1 = memory.add("...", user_id=user_id)
# print(resultat_1)


In [ ]:
# TODO — étape 3 : message qui enrichit le fait précédent


In [ ]:
# TODO — étape 4 : message qui contredit le fait initial


In [ ]:
# TODO — étape 5 : message redondant (NOOP attendu)


In [ ]:
# TODO — affiche l'état final de la mémoire de cet utilisateur avec memory.get_all(...)
# et vérifie que ce que tu observes correspond à ce que tu attendais.


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Ta réponse ici :_

> 

**❓ Question ouverte** — As-tu réussi à provoquer un `NOOP` du premier coup ? Si non, qu'est-ce que ça t'apprend sur la façon dont le LLM sous-jacent juge la "nouveauté" d'une information ?

_Ta réponse ici :_

> 

## Item 3 : Persistance single-hop

Objectif : reproduire, avec tes propres mots, le scénario du papier de recherche (l'exemple "végétarien" — un utilisateur donne une contrainte, puis revient plus tard demander une recommandation qui doit la respecter).

**Consignes :**
1. Écris une fonction `repondre_avec_memoire(user_id, question)` qui :
   - récupère les souvenirs pertinents via `memory.search(...)`,
   - construit un prompt qui inclut ce contexte,
   - appelle le LLM (`OpenAI` ou `ChatOpenAI`) pour générer une réponse,
   - retourne cette réponse.
2. Écris aussi une fonction `repondre_sans_memoire(question)` qui n'utilise **aucun** contexte.
3. Teste les deux fonctions sur la même question de suivi, après avoir enregistré une contrainte au préalable.


In [ ]:
from openai import OpenAI
client = OpenAI()

def repondre_avec_memoire(user_id: str, question: str) -> str:
    """TODO : documente ici ce que fait ta fonction, étape par étape."""
    # TODO
    pass

def repondre_sans_memoire(question: str) -> str:
    """TODO"""
    # TODO
    pass


In [ ]:
# TODO — enregistre une contrainte (régime, préférence, contrainte de planning, ce que tu veux)


In [ ]:
# TODO — pose la même question de suivi aux deux fonctions et affiche les deux réponses côte à côte


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Ta réponse ici :_

> 

## Item 4 : Mémoire multi-hop et raisonnement temporel

**Consignes :**
1. Simule au moins 3 "sessions" espacées dans le temps pour un même utilisateur (utilise `time.sleep(...)` entre chaque `memory.add`), chacune apportant un fait distinct mais reliable aux autres (ex : un contexte professionnel qui évolue, un projet qui avance par étapes...).
2. Pose une question **multi-hop** : une question dont la réponse nécessite de combiner au moins deux faits enregistrés séparément.
3. Pose une question **temporelle** : une question qui demande de savoir *quand* ou *dans quel ordre* les choses se sont passées.
4. Pour chaque question, indique **avant** de l'exécuter ce que tu attends comme réponse correcte, puis compare avec ce que le système a réellement produit.


In [ ]:
# TODO — construis tes 3+ sessions espacées dans le temps


In [ ]:
# TODO — question multi-hop : formule-la, note ta réponse attendue en commentaire, puis interroge le système


In [ ]:
# TODO — question temporelle : même démarche


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Indice pour l'interprétation : le papier de recherche note que Mem0 (mémoire en langage naturel) et Mem0ᵍ (mémoire en graphe) n'ont pas les mêmes forces selon le type de question. Est-ce cohérent avec ce que tu observes ?_

_Ta réponse ici :_

> 

## Item 5 : Mini LLM-as-a-Judge

**Consignes :**
1. Écris une fonction `juger_reponse(question, reponse_attendue, reponse_generee)` qui utilise un LLM pour juger si la réponse générée est correcte. **Rédige toi-même le prompt d'évaluation** (ne le copie d'aucune source) — réfléchis à ce qui doit être précisé pour que le jugement soit fiable (tolérance de formulation, gestion des dates relatives, etc., comme discuté dans le résumé du papier).
2. Construis un mini jeu de test d'au moins 6 questions couvrant les 4 catégories vues dans le papier (single-hop, multi-hop, temporal, open-domain).
3. Calcule un taux de réussite global et un taux par catégorie.


In [ ]:
def juger_reponse(question: str, reponse_attendue: str, reponse_generee: str) -> dict:
    """TODO : écris ton propre prompt de jugement. Réfléchis à :
    - comment gérer une réponse plus longue mais correcte,
    - comment gérer les formulations de dates différentes,
    - le format de sortie que tu veux (JSON recommandé pour pouvoir le parser).
    """
    # TODO
    pass


In [ ]:
# TODO — construis ton jeu de test structuré, par exemple une liste de dicts avec
# {"categorie": "single-hop", "user_id": ..., "question": ..., "reponse_attendue": ...}


In [ ]:
# TODO — boucle d'évaluation : génère une réponse, juge-la, stocke le verdict, puis calcule les taux par catégorie


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Sur quelle(s) catégorie(s) ton système se trompe-t-il le plus ? Donne au moins un exemple concret d'échec avec la réponse générée et pourquoi elle est fausse._

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Ta réponse ici :_

> 

## Item 6 : Mémoire en graphe (Mem0ᵍ) (bonus)

Nécessite une instance Neo4j (locale via Docker, ou Neo4j Aura Free) :

```bash
docker run -d -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:latest
```

**Consignes :**
1. Configure une seconde instance `Memory` avec un `graph_store` Neo4j.
2. Réutilise le jeu de faits impliquant plusieurs entités reliées (personnes, lieux, événements) de l'item 4.
3. Compare la réponse obtenue à une question relationnelle avec la mémoire classique vs la mémoire en graphe.


In [ ]:
# TODO — configuration Mem0 avec graph_store Neo4j (adapte url/username/password)


In [ ]:
# TODO — même question posée aux deux mémoires (classique vs graphe), affichage côte à côte


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Ta réponse ici :_

> 

## Item 7 : Intégration LangChain

**Consignes :**
1. Crée une classe qui hérite de `langchain_core.chat_history.BaseChatMessageHistory`, dont les méthodes `add_message` et `messages` s'appuient respectivement sur `memory.add(...)` et `memory.search(...)`.
2. Branche cette classe dans un `RunnableWithMessageHistory` autour d'un simple prompt + LLM.
3. Teste une conversation sur plusieurs tours et vérifie que le contexte est bien conservé entre les appels, **sans** que tu aies à repasser l'historique manuellement.

Documentation à consulter toi-même : https://python.langchain.com/docs/how_to/message_history/


In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage

class Mem0ChatMessageHistory(BaseChatMessageHistory):
    """TODO : documente le rôle de chaque méthode que tu implémentes."""

    def __init__(self, user_id: str):
        # TODO
        pass

    @property
    def messages(self) -> list:
        # TODO : doit retourner une liste de BaseMessage reconstruite depuis Mem0
        pass

    def add_message(self, message: BaseMessage) -> None:
        # TODO : doit écrire ce message dans Mem0
        pass

    def clear(self) -> None:
        # TODO
        pass


In [ ]:
# TODO — assemble ta chaîne LangChain (prompt + llm) avec RunnableWithMessageHistory
# puis fais 2-3 tours de conversation successifs pour vérifier la persistance du contexte


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Qu'est-ce que cette intégration t'apporte par rapport à l'appel manuel de `memory.search`/`memory.add` que tu faisais en partie 3 ? Quels sont les inconvénients éventuels ?_

_Ta réponse ici :_

> 

## Item 8 : Agent LangGraph avec mémoire persistante

**Consignes :**
1. Définis un état (`TypedDict`) contenant au minimum : les messages de la conversation, l'identifiant utilisateur, et le contexte mémoire récupéré.
2. Construis un graphe (`StateGraph`) avec au moins 3 nœuds : récupération de mémoire → génération de réponse → écriture en mémoire.
3. Compile et teste ton agent sur au moins deux tours de conversation.
4. Dessine ou décris (en commentaire) le graphe que tu as construit, comme si tu devais l'expliquer à quelqu'un qui ne connaît pas LangGraph.


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    # TODO : définis les champs de ton état
    pass

def noeud_recuperer_memoire(state: AgentState) -> dict:
    """TODO"""
    pass

def noeud_generer_reponse(state: AgentState) -> dict:
    """TODO"""
    pass

def noeud_ecrire_memoire(state: AgentState) -> dict:
    """TODO"""
    pass

# TODO — assemble le graphe : add_node, set_entry_point, add_edge, compile()


In [ ]:
# TODO — teste l'agent sur au moins deux tours successifs (deux .invoke() qui partagent le même user_id)


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Ta réponse ici :_

> 

## Item 9 : Connexion d'un outil (agent SAV Beqo)

Le périmètre de cet agent n'est pas à choisir librement : tu vas construire un agent de service après-vente pour l'agence de voyage **Beqo**, dont la mission unique est la **gestion des réclamations et des plaintes clients**. Rien d'autre.

**Consignes :**
1. Définis un mini schéma de données pour une réclamation (ex. : identifiant client, objet du voyage concerné, nature de la plainte, statut).
2. Écris la fonction qui enregistre une réclamation, puis déclare-la comme `@tool` LangChain avec une docstring précise sur le périmètre strict de l'agent (réclamations uniquement) et sur le moment où l'outil doit être déclenché.
3. Branche cet outil dans ton agent LangGraph via `bind_tools` + `ToolNode` + une route conditionnelle (`tools_condition`), en veillant à ce que le graphe reboucle vers le nœud de génération après l'appel.
4. Teste trois cas : un message qui est clairement une réclamation (ex. bagage perdu, vol annulé, remboursement non reçu) et **doit** déclencher l'outil ; un message hors périmètre (ex. nouvelle réservation) qui **ne doit pas** le déclencher ; un message ambigu, à la frontière entre les deux (ex. question sur les conditions d'annulation) — c'est ce troisième cas qui révèle si le périmètre de ton agent est bien défini.


In [ ]:
from langchain_core.tools import tool

# TODO — mini schéma de données pour une réclamation (dict, dataclass ou pydantic, à toi de choisir)
# class Reclamation:
#     ...

@tool
def enregistrer_reclamation(...) -> str:
    """TODO : docstring précise sur le périmètre de l'agent SAV Beqo (réclamations uniquement)
    et sur le moment où cet outil doit être déclenché."""
    pass


In [ ]:
# TODO — reconstruis ton graphe LangGraph avec bind_tools + ToolNode + tools_condition


In [ ]:
# TODO — 3 tests : (1) doit déclencher l'outil, (2) ne doit pas, (3) mémoire + outil combinés


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Dans le test (2), l'agent a-t-il bien évité d'appeler l'outil inutilement ? Si non, qu'aurais-tu pu changer dans la docstring de l'outil pour clarifier son usage ?_

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Ta réponse ici :_

> 

## Item 10 : Mesure de latence, coût en tokens, comparaison à un baseline

**Consignes :**
1. Écris une fonction `repondre_contexte_complet(user_id, question)` qui récupère **tout** l'historique brut (`memory.get_all`) et le passe intégralement au LLM, sans passer par `memory.search`.
2. Sur un même jeu de 5 questions, mesure pour chaque approche (`repondre_avec_memoire` vs `repondre_contexte_complet`) :
   - la latence (`time.perf_counter()`),
   - le nombre de tokens envoyés au LLM (`tiktoken`).
3. Affiche un graphique comparatif (barres) des deux approches.
4. Mets ces résultats en regard des chiffres du papier de recherche (91 % de réduction de latence p95, plus de 90 % de réduction de tokens) : es-tu dans le même ordre de grandeur ? Si non, pourquoi, à ton avis (taille de ta base de mémoire, longueur des conversations testées...) ?


In [ ]:
import time
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def repondre_contexte_complet(user_id: str, question: str) -> str:
    """TODO"""
    pass

def mesurer(fonction, *args, **kwargs):
    """TODO : chronomètre l'exécution de `fonction` et retourne (résultat, durée)"""
    pass


In [ ]:
# TODO — boucle de mesure sur ton jeu de 5 questions, stocke durées et tokens pour chaque approche


In [ ]:
import matplotlib.pyplot as plt

# TODO — trace un graphique en barres comparant les deux approches (latence ET tokens, 2 sous-graphiques par exemple)


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Compare tes chiffres à ceux du papier de recherche. Explique au moins un facteur qui pourrait expliquer un écart entre tes résultats et les leurs._

_Ta réponse ici :_

> 

## Item 11 : Bilan

**À rédiger toi-même, en quelques paragraphes :**

1. Résume, avec tes propres mots (pas ceux de l'article), ce qu'apporte concrètement une architecture comme Mem0 par rapport à un simple historique de conversation passé en entier au LLM.
2. Sur quel(s) item(s) as-tu observé un comportement inattendu ? Qu'as-tu appris de cet écart entre attente et résultat ?
3. Pour ton projet d'agent WhatsApp business : identifie 2 ou 3 endroits précis où tu comptes réutiliser un des mécanismes vus ici (mémoire multi-utilisateurs, outil externe, mesure de latence...), et explique pourquoi.


_Ta réponse ici :_

> 